# Website Summarizer (Local LLaMA via Ollama)

This notebook demonstrates:
- Fetching website content
- Cleaning HTML
- Summarizing using a local LLM (LLaMA 3.2 1B)
- Adding a fun fact for engagement

No API required — fully local inference.

In [1]:
!ollama pull llama3.2:1b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 74701a8c35f6: 100% ▕██████████████████▏ 1.3 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 4f659a1e86d7: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


In [2]:
import requests
from bs4 import BeautifulSoup
import ollama
from IPython.display import Markdown, display

In [3]:
headers = {"User-Agent": "Mozilla/5.0"}
def fetch_website_contents(url):
    response = requests.get(url, headers=headers)
    soup =BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title"
    for tag in soup(["script", "style", "img", "input"]):
        tag.decompose()
    text = soup.get_text(separator="\n", strip=True)

    return (title + "\n\n" + text)[:2000]

In [4]:
def build_prompt(content):
    return f"""
You are a smart and slightly funny assistant.

Tasks:
1. Summarize the website content in 3-5 bullet points
2. Ignore navigation/menu text
3. Add 1 interesting or fun fact related to the topic

Website content:
{content}
"""

In [5]:
def call_local_llama(prompt):
    response = ollama.chat(
        model="llama3.2:1b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response["message"]["content"]

In [6]:
def summarize_website(url):
    content = fetch_website_contents(url)
    prompt = build_prompt(content)
    return call_local_llama(prompt)

In [7]:
def display_summary(url):
    summary = summarize_website(url)
    display(Markdown(summary))

In [8]:
display_summary("https://cnn.com")

I'll ignore navigation/menu text and provide a summary of the website content in 3-5 bullet points, along with an interesting or fun fact related to the topic. Here's the summary:

• The website is a news aggregator that provides breaking news, latest news, and videos from CNN (Channel CNN). It seems to be a one-stop-shop for news and information.
• The content is categorized by topic, including Politics, Business, Health, Entertainment, Style, Travel, Sports, Science, Climate, and more. This suggests that the website aims to provide a broad range of topics and news.
• The website appears to be a general news platform, but I couldn't find any specific information about its editorial independence or journalistic standards. It seems to be a commercial venture that relies on advertising revenue.
• One fun fact I found is that CNN has a "CNN Polls" section, which suggests that the website may be using the polls as a way to engage with its audience and provide a sense of community. This might be an interesting feature for those who enjoy opinion-driven content.

Overall, the website seems to be a general news platform that aggregates content from various sources, including CNN. It may be a useful resource for those looking for news and information on a wide range of topics.

In [9]:
def build_prompt(content, mode="funny"):
    if mode == "funny":
        style = "Be witty and humorous 😄"
    elif mode == "professional":
        style = "Be concise and professional"
    elif mode == "eli5":
        style = "Explain like I'm 5 years old"
    elif mode == "dramatic":
        style = "Make it dramatic and exciting 🔥"
    else:
        style = "Be clear"

    return f"""
You are an AI assistant.

Tasks:
1. Summarize the website in 3-5 bullet points
2. Add 1 interesting fact
3. {style}

Website content:
{content}
"""

In [10]:
def compare_styles(url):
    content = fetch_website_contents(url)

    modes = ["funny", "professional", "eli5"]

    for mode in modes:
        print(f"\n🧠 Mode: {mode.upper()}")
        prompt = build_prompt(content, mode)
        result = call_local_llama(prompt)
        display(Markdown(result))

In [11]:
compare_styles("https://cnn.com")


🧠 Mode: FUNNY


Here are 3-5 bullet points summarizing the website:

• **The ultimate news destination**: CNN is your go-to source for breaking news, latest updates, and videos from around the world.
• **Content overload**: The website has several issues, including slow video player loading, audio on ads being too loud, and repetitive ad content.
• **Technical troubles**: Users have reported problems with the video player freezing, the ad not finishing loading, and content being moved around while loading.
• **Feedback is appreciated**: CNN values your feedback and is always looking for ways to improve the user experience.

And here's a witty and humorous addition:

"Sorry, folks, but CNN's ad system is like a bad boyfriend: it's always freezing on you, being too loud, and making you feel like you're stuck in the 'ad fatigue' cycle. But hey, at least you get to see a lot of the same ads over and over again, right?"


🧠 Mode: PROFESSIONAL


Here are 3-5 bullet points summarizing the website:

• CNN provides news, breaking news, and videos in various categories including politics, business, health, entertainment, and more.
• The website has a video player that occasionally experiences issues such as slow load times, ad freezes, and audio volume issues.
• Users have reported a range of technical issues, including content movement while ads load, repetitive ads, and other problems.
• The website also receives feedback and support from users, who can cancel their account or submit feedback.

One interesting fact about the website is that CNN's "CNN Polls" and "2026 Elections" sections are subject to periodic downtime, highlighting the website's ongoing efforts to provide accurate and reliable information.


🧠 Mode: ELI5


Here are the requested tasks:

**1. Summarize the website in 3-5 bullet points:**

* CNN is a news website where you can find breaking news, latest news, and videos.
* The website has various sections like Politics, Business, Health, Entertainment, and Sports.
* You can search for specific topics, such as the Ukraine-Russia War, Israel-Hamas War, or the 2026 Elections.
* The website also has a feature called "Facts First" where they try to provide accurate information about the topics they cover.

**2. Add 1 interesting fact:**

* The name "CNN" stands for "Channel News Network" and was founded in 1980.

**3. Explain like I'm 5 years old:**

Imagine you're watching TV, and you want to learn more about something. You go to a website called CNN, and it has lots of news about what's happening in the world. You can watch videos about important events, like wars or elections. The website has special features, like "Facts First", that help you learn more about the topics you care about. It's like having a special helper that gives you the news you want to know!

**4. What is "Ad Feedback" section?**

* The "Ad Feedback" section is where you can tell CNN how the ads on their website are doing.
* You can rate the ads as "good", "bad", or "meh" and tell them what you think about them.
* This feedback helps CNN to make their ads better and more relevant to you in the future.

**5. "Other issues" section:**

* This section mentions some other problems that people have encountered on the website.
* They include things like ads freezing or not loading properly, video players being slow to load, and content moving around while the ad is playing.
* These issues can be frustrating, but CNN's website has a "Help" section where you can find solutions to these problems.

In [14]:
def fun_fact(url):
    content = fetch_website_contents(url)

    prompt = f"""
Read this content and give ONLY one surprising or fun fact:

{content}
"""
    return call_local_llama(prompt)

fun_fact("https://cnn.com")

"Here's a surprising or fun fact from the content:\n\n1. Video content was moved around while it loaded to accommodate a different video player, resulting in the ad freezing or not finishing loading."